In [ ]:
# ===============================
# LSTM-Based Text Prediction System
# Group Assignment - Deployment: FastAPI
# ===============================

# -------------------------------
# 1. Dataset Collection (Using Wikipedia API)
# -------------------------------
# Dataset Name: Wikipedia Text Corpus
# Source: Wikipedia API (via `wikipedia` library)
# Description: Short article summaries for next-word prediction

!pip install wikipedia-api

import wikipediaapi
import re

# Fetch Wikipedia articles
# FIX: Specify a user_agent as required by Wikipedia API
wiki_wiki = wikipediaapi.Wikipedia(user_agent='MyTextPredictionApp/1.0 (https://example.com/my-app)', language='en')
topics = ['Artificial intelligence', 'Machine learning', 'Deep learning', 'Neural network', 'LSTM']

raw_text = ""
for topic in topics:
    page = wiki_wiki.page(topic)
    if page.exists():
        raw_text += page.text + "\n"

# Save raw text
with open('corpus.txt', 'w', encoding='utf-8') as f:
    f.write(raw_text)

print(f"Total characters collected: {len(raw_text)}")

Total characters collected: 243693


In [ ]:
# -------------------------------
# 2. Data Preprocessing
# -------------------------------
!pip install tensorflow # Install tensorflow
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Clean text
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

cleaned_corpus = clean_text(raw_text)

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts([cleaned_corpus])
total_words = len(tokenizer.word_index) + 1

print(f"Total unique words: {total_words}")

# Create sequences
input_sequences = []
for line in cleaned_corpus.split('. '):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences
# FIX: Cap max_sequence_len to avoid GPU OOM errors due to excessively long sequences
# A typical max_sequence_len for next-word prediction is often between 50-100.
original_max_sequence_len = max([len(seq) for seq in input_sequences])
max_sequence_len = min(original_max_sequence_len, 50) # Cap at 50 for demonstration/memory purposes

input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')

# Create predictors and label
X, y = input_sequences[:, :-1], input_sequences[:, -1]
y = np.array(y)

print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"Original max sequence length: {original_max_sequence_len}, Capped max sequence length: {max_sequence_len}")


Total unique words: 5328
X shape: (32232, 49), y shape: (32232,)
Original max sequence length: 32233, Capped max sequence length: 50


In [ ]:
# The previous command failed because 'iJv66O02ImMQ' is a cell ID, not a file.
# To apply the preprocessing changes and prevent GPU crashes:
# 1. Scroll up to the cell with ID 'iJv66O02ImMQ' (Section 2. Data Preprocessing).
# 2. Click the Run button on that cell.
# 3. Then proceed to training the model in the following cells.

In [ ]:
import tensorflow as tf

# Check for GPU availability
gpu_available = tf.config.list_physical_devices('GPU')

if gpu_available:
    print(f"GPU is available: {gpu_available}")
    print("Model training should utilize the GPU.")
else:
    print("No GPU devices found. Training will use CPU.")
    print("Please ensure you have selected a GPU runtime in Colab (Runtime -> Change runtime type).")

GPU is available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Model training should utilize the GPU.


In [ ]:
# -------------------------------
# 3. LSTM Model Development
# -------------------------------
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

# One-hot encode labels
y = to_categorical(y, num_classes=total_words)

# Build LSTM model
model = Sequential()
model.add(Embedding(total_words, 100))
model.add(LSTM(150, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dropout(0.2))
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# Train model
history = model.fit(X, y, epochs=50, verbose=1, validation_split=0.2)

# Save model
model.save('lstm_text_predictor.h5')
print("Model saved as lstm_text_predictor.h5")

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
806/806 ━━━━━━━━━━━━━━━━━━━━ 13s 13ms/step - accuracy: 0.0465 - loss: 7.0395 - val_accuracy: 0.0566 - val_loss: 7.3223
Epoch 2/50
806/806 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.0634 - loss: 6.6380 - val_accuracy: 0.0656 - val_loss: 7.3860
Epoch 3/50
806/806 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.0736 - loss: 6.4690 - val_accuracy: 0.0731 - val_loss: 7.4094
Epoch 4/50
806/806 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.0843 - loss: 6.3096 - val_accuracy: 0.0800 - val_loss: 7.5097
Epoch 5/50
806/806 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.0982 - loss: 6.1484 - val_accuracy: 0.0869 - val_loss: 7.5721
Epoch 6/50
806/806 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.1091 - loss: 5.9956 - val_accuracy: 0.0929 - val_loss: 7.7368
Epoch 7/50
806/806 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.1098 - loss: 5.9465 - val_accuracy: 0.0875 - val_loss: 7.8043
Epoch 8/50
806/806 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.1215 - loss: 5.7607 - v

Model saved as lstm_text_predictor.h5


In [ ]:
# -------------------------------
# 4. Prediction Function
# -------------------------------
def predict_next_word(seed_text, model, tokenizer, max_sequence_len, top_n=3):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    if len(token_list) >= max_sequence_len:
        token_list = token_list[-(max_sequence_len-1):]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted_probs = model.predict(token_list, verbose=0)[0]

    # Get top N predictions
    top_indices = np.argsort(predicted_probs)[-top_n:][::-1]
    predictions = []
    for idx in top_indices:
        for word, index in tokenizer.word_index.items():
            if index == idx:
                predictions.append(word)
                break
    return predictions

# Test prediction
seed = "artificial intelligence is"
next_words = predict_next_word(seed, model, tokenizer, max_sequence_len)
print(f"Seed: '{seed}'")
print(f"Next word predictions: {next_words}")

Seed: 'artificial intelligence is'
Next word predictions: ['going', 'difficult', 'a']


In [ ]:
# -------------------------------
# 5. FastAPI Deployment Code (Generate API file)
# -------------------------------
api_code = """
from fastapi import FastAPI, HTTPRequest
from pydantic import BaseModel
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
import pickle
import re

app = FastAPI(title="LSTM Text Prediction API")

# Load model and tokenizer
model = load_model('lstm_text_predictor.h5')

# Load tokenizer (you'll need to save it during training)
with open('tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

max_sequence_len = {max_sequence_len}  # Replace with actual value

class TextInput(BaseModel):
    seed_text: str
    top_n: int = 3

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\\s]', '', text)
    text = re.sub(r'\\s+', ' ', text).strip()
    return text

def predict_next_word(seed_text, model, tokenizer, max_sequence_len, top_n=3):
    seed_text = clean_text(seed_text)
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    if len(token_list) >= max_sequence_len:
        token_list = token_list[-(max_sequence_len-1):]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted_probs = model.predict(token_list, verbose=0)[0]

    top_indices = np.argsort(predicted_probs)[-top_n:][::-1]
    predictions = []
    for idx in top_indices:
        for word, index in tokenizer.word_index.items():
            if index == idx:
                predictions.append(word)
                break
    return predictions

@app.get("/")
def root():
    return {"message": "LSTM Text Prediction API is running"}

@app.post("/predict")
def predict(input: TextInput):
    predictions = predict_next_word(
        input.seed_text,
        model,
        tokenizer,
        max_sequence_len,
        input.top_n
    )
    return {
        "seed_text": input.seed_text,
        "predictions": predictions,
        "top_n": input.top_n
    }
"""

# Save tokenizer for deployment
import pickle
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

# Save API code
with open('main.py', 'w') as f:
    f.write(api_code.replace("{max_sequence_len}", str(max_sequence_len)))

print("API code saved as main.py")
print("Tokenizer saved as tokenizer.pkl")

API code saved as main.py
Tokenizer saved as tokenizer.pkl


In [ ]:
# -------------------------------
# 6. Requirements.txt for Deployment
# -------------------------------
requirements = """
fastapi
uvicorn
tensorflow
numpy
pickle-mixin
re
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print("requirements.txt created")

requirements.txt created


In [ ]:
# -------------------------------
# 7. Testing Instructions
# -------------------------------
print("\n" + "="*50)
print("DEPLOYMENT INSTRUCTIONS")
print("="*50)
print("""
To deploy the FastAPI locally:
1. Download these files: lstm_text_predictor.h5, tokenizer.pkl, main.py, requirements.txt
2. Install dependencies: pip install -r requirements.txt
3. Run: uvicorn main:app --reload
4. Test API:
   - Swagger UI: http://localhost:8000/docs
   - POST to /predict with JSON:
     {
         "seed_text": "artificial intelligence is",
         "top_n": 3
     }

To deploy on Render / Hugging Face Spaces:
- Create a new Web Service on Render
- Set build command: pip install -r requirements.txt
- Set start command: uvicorn main:app --host 0.0.0.0 --port $PORT
- Upload all files to GitHub and connect
""")


DEPLOYMENT INSTRUCTIONS

To deploy the FastAPI locally:
1. Download these files: lstm_text_predictor.h5, tokenizer.pkl, main.py, requirements.txt
2. Install dependencies: pip install -r requirements.txt
3. Run: uvicorn main:app --reload
4. Test API:
   - Swagger UI: http://localhost:8000/docs
   - POST to /predict with JSON:
     {
         "seed_text": "artificial intelligence is",
         "top_n": 3
     }

To deploy on Render / Hugging Face Spaces:
- Create a new Web Service on Render
- Set build command: pip install -r requirements.txt
- Set start command: uvicorn main:app --host 0.0.0.0 --port $PORT
- Upload all files to GitHub and connect



In [ ]:
# -------------------------------
# 8. Sample API Test (Simulated)
# -------------------------------
import requests
import json

# Simulated API test (if running locally after deployment)
print("\nSample API Response Simulation:")
sample_input = {"seed_text": "deep learning is", "top_n": 3}
preds = predict_next_word("deep learning is", model, tokenizer, max_sequence_len)
print(f"Input: {sample_input['seed_text']}")
print(f"Predictions: {preds}")


Sample API Response Simulation:
Input: deep learning is
Predictions: ['a', 'the', 'not']
